### Vector Store is a component that stores vectors and lets you retrieve similar ones.
### Vector Database is a complete database system designed specifically for vector data.

| Feature            | Vector Store       | Vector Database          |
| ------------------ | ------------------ | ------------------------ |
| Scope              | Narrow (component) | Full system              |
| Persistence        | Optional / limited | Built-in                 |
| Scaling            | Limited            | Designed for scale       |
| Metadata filtering | Basic or none      | Advanced                 |
| Indexing           | Sometimes basic    | Advanced (HNSW, IVF, PQ) |
| APIs               | Minimal            | Full REST/gRPC           |
| Production ready   | Not always         | Yes                      |


# RAG system with LangChain and ChromaDB

In [2]:
import os
from dotenv import load_dotenv
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain.chat_models.base import init_chat_model
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

True

In [3]:
# direct SDK-style wrapper for OpenAI:
# llm = ChatOpenAI(
#     model="gpt-3.5-turbo",
#     # temperature=0.2,
#     max_tokens=500
# )

# class / provider abstraction layer that can dynamically select the provider (OpenAI, Anthropic, Azure, etc.)
llm = init_chat_model("openai:gpt-3.5-turbo")

# TEST:
# test_llm = llm.invoke("What is a Large Language Model?")
# print(test_llm)

embedding_model = "text-embedding-3-small"
embedding = OpenAIEmbeddings(model=embedding_model)

In [4]:
pypdf_doc = []

try:
    pypdf_loader = PyPDFLoader("../DataIngestParsing/data/pdf/Privacy_Policy_roadRecorder.pdf")
    pypdf_doc = pypdf_loader.load()

    full_text = "\n\n".join(page.page_content for page in pypdf_doc)
    metadata = pypdf_doc[-1].metadata

    # text_splitter = RecursiveCharacterTextSplitter(
    #     chunk_size=150,       # large enough to never cut mid-paragraph (the number determine by Avg + extra for safe celling)
    #     chunk_overlap=0,      # no overlap needed on self-contained paragraphs
    #     separators=["\n"]     # split on paragraph breaks only
    # )

    # print(text_splitter)

    print(pypdf_doc[0])
    print(len(pypdf_doc))
    # print(pypdf_doc)
except Exception as e:
    print(f"Error loading PyPDF: {e}")

page_content='Privacy Policy for roadRecorder 
 
Privacy Policy 
Last updated: September 29, 2024 
This Privacy Policy describes Our policies and procedures on the collecƟon, use and 
disclosure of Your informaƟon when You use the Service and tells You about Your privacy 
rights and how the law protects You. 
We use Your Personal data to provide and improve the Service. By using the Service, You 
agree to the collecƟon and use of informaƟon in accordance with this Privacy Policy. This 
Privacy Policy has been created with the help of the Privacy Policy Generator. 
 
InterpretaƟon and DeﬁniƟons 
InterpretaƟon 
The words of which the iniƟal leƩer is capitalized have meanings deﬁned under the 
following condiƟons. The following deﬁniƟons shall have the same meaning regardless of 
whether they appear in singular or in plural. 
DeﬁniƟons 
For the purposes of this Privacy Policy: 
 Account means a unique account created for You to access our Service or parts of 
 our Service. 
 Aﬃliate me

#### Create a ChromaDB vector store:
with from_documents we can open store and load chunks at the same time

In [4]:
persist_directory = "./chroma_db"
vector_store = Chroma.from_documents(
    documents=pypdf_doc,
    embedding=embedding,
    persist_directory=persist_directory, # This will create chroma_db folder for the vector store with all the data
    collection_name="rag_collection"
)

print(f"Created vector store with {len(vector_store)} vectors")

Created vector store with 1 vectors


#### Similarity Test

In [5]:
query = "How can I contact with the app owner?"

similar_docs = vector_store.similarity_search(query, k=2)
similar_docs_with_scores = vector_store.similarity_search_with_score(query, k=2)

# scores with ChromaDB:
# Closer to 0 = Similar result
# Zero means identical vectors

print(similar_docs[0])
print("---------")
print(similar_docs_with_scores[0])

page_content='If you have any quesƟons about this Privacy Policy, You can contact us: 
By email: idox2x@gmail.com 
Generated using TermsFeed Priv 
acy Policy Generator' metadata={'page_label': '9', 'source': '../DataIngestParsing/data/pdf/Privacy_Policy_roadRecorder.pdf', 'page': 8, 'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'moddate': '2024-10-02T14:21:09+03:00', 'creationdate': '2024-10-02T14:21:09+03:00', 'title': 'Microsoft Word - New Microsoft Word Document', 'author': 'Ido madaRR', 'total_pages': 9}
---------
(Document(metadata={'source': '../DataIngestParsing/data/pdf/Privacy_Policy_roadRecorder.pdf', 'creator': 'PyPDF', 'title': 'Microsoft Word - New Microsoft Word Document', 'total_pages': 9, 'creationdate': '2024-10-02T14:21:09+03:00', 'page_label': '9', 'page': 8, 'moddate': '2024-10-02T14:21:09+03:00', 'producer': 'Microsoft: Print To PDF', 'author': 'Ido madaRR'}, page_content='If you have any quesƟons about this Privacy Policy, You can contact us: \nBy ema

# RAG Chain

#### Approach 1 — High-level API

The goal of the chain is:

* Takes retrieved documents
* Stuff them into the prompt's context placeholder
* Sends the complete prompt to the LLM
* Return the LLm's response

In [53]:
# Converting vector store to retriever

retriever = vector_store.as_retriever(
   search_kwargs={"k": 2} # retrieve 3 relevant chunks
)

retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001FF3FD64050>, search_kwargs={'k': 2})

In [54]:
# Prompt Template

system_prompt = """
    You are assistant for question-answering tasks.
    User the following pieces of retrieved context to answer the question.
    Make sure to keep your answer concise.
    If you don't know the answer, just say that you don't know.

    Context: {context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])

prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\n    You are assistant for question-answering tasks.\n    User the following pieces of retrieved context to answer the question.\n    Make sure to keep your answer concise.\n    If you don't know the answer, just say that you don't know.\n\n    Context: {context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [55]:
# Building the document chain
# The document_chain knows how to answer a question given some context, but it has no way to find that context on its own. The retriever knows how to search the vector store and pull relevant chunks, but it can't produce an answer. create_retrieval_chain connects them — the retriever fetches the relevant documents, and passes them as context into the document chain, which then answers the question.

document_chain = create_stuff_documents_chain(llm, prompt)
print(document_chain)

print("---------------")

# Building the rag chain
# create_retrieval_chain — the full pipeline - This chain wraps the first one and adds the retrieval step in front
rag_chain = create_retrieval_chain(retriever, document_chain)
print(rag_chain)

bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\n    You are assistant for question-answering tasks.\n    User the following pieces of retrieved context to answer the question.\n    Make sure to keep your answer concise.\n    If you don't know the answer, just say that you don't know.\n\n    Context: {context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOpenAI(output_version=None, profile={'name': 'GPT-3.5-turbo', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False

In [57]:
response = rag_chain.invoke({
    "input": "How can I contact with the app owner?",
})

print(response.get("answer"))

You can contact the app owner by email at idox2x@gmail.com.


####  Approach 2 — LCEL (LangChain Expression Language)

In [23]:
custom_prompt = ChatPromptTemplate.from_template(
    """
        Use the following pieces of retrieved context to answer the question.
        If you don't know the answer, just say that you don't know.
        If the question if witten on Hebrew, return the answer in hebrew.
        Provide specific details from the context to answer the question.

        Context: {context}

        Question: {question}
    """
)

custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\n        User the following pieces of retrieved context to answer the question.\n        If you don't know the answer, just say that you don't know.\n        If the question if witten on Hebrew, return the answer in hebrew.\n        Provide specific details from the context to answer the question.\n\n        Context: {context}\n\n        Question: {question}\n    "), additional_kwargs={})])

In [20]:
retriever = vector_store.as_retriever(
   search_kwargs={"k": 2} # retrieve 3 relevant chunks
)



In [24]:
#  It takes that a list of retrieved documents and converts it into one single string:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain_lcel=(
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    } | custom_prompt | llm | StrOutputParser()
)

In [25]:
rag_chain_lcel.invoke("Which kind of personal data the app collects?")

'The app collects Usage Data such as Internet Protocol address, browser type, browser version, pages visited, time and date of visit, time spent on pages, unique device identifiers, and other diagnostic data. It also collects information related to mobile devices such as the type of device, unique ID, IP address, operating system, Internet browser type, and other diagnostic data. Additionally, with prior permission, the app may collect "background location" information to provide features of the service.'

#### Add new document

In [15]:

another_pypdf_doc = []

try:
    pypdf_loader = PyPDFLoader("../DataIngestParsing/data/pdf/terms_and_condition.pdf")
    pages = pypdf_loader.load()

    # Merge all pages into one text before splitting.
    full_text = "\n\n".join(page.page_content for page in pages)

    paragraphs = [p for p in full_text.split("\n") if p.strip()]
    lengths = [len(p) for p in paragraphs]
    print(f"Min: {min(lengths)}, Max: {max(lengths)}, Avg: {sum(lengths)//len(lengths)}")

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=150,       # large enough to never cut mid-paragraph (the number determine by Avg + extra for safe celling)
        chunk_overlap=0,      # no overlap needed on self-contained paragraphs
        separators=["\n"]     # split on paragraph breaks only
    )

    new_chunks = text_splitter.create_documents([full_text])
    print(len(new_chunks))
    print(new_chunks[0])

    vector_store.add_documents(new_chunks)

except Exception as e:
    print(f"Error loading PyPDF: {e}")



# vector_store.add_documents()

Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 38 0 (offset 0)
Ignoring wrong pointing object 55 0 (offset 0)
Ignoring wrong pointing object 56 0 (offset 0)


Min: 5, Max: 110, Avg: 76
301
page_content='צעיר הפניקס אפליקציית שימוש תנאי
 :כללי
.1 ״הפניקס הביטוח תכנית של הרכב באפליקציית שימוש לעשות שבחרת שמחים אנו, יקר מבוטח'


In [25]:
# print(f"Created vector store with {len(vector_store)} vectors")

rag_chain_lcel.invoke("האם החברה רשאית לתת הטבות?")


'כן, החברה רשאית לתת הטבות אם קיימת הסכמה מהצד של הלקוחות.'